# Smart QC Pipeline - Enhanced with CV

**Features:**
- Improved field extraction patterns based on OPUS rules
- Soft CV integration for photo-based validation
- Strict validation for non-CV rules
- Comprehensive convinced scoring

## 1. Install Dependencies

In [ ]:
# Install Tesseract OCR
!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-eng

# Install Python packages (including CV dependencies)
!pip install -q pytesseract spacy pandas pydantic sentence-transformers PyMuPDF httpx openpyxl opencv-python-headless numpy

print("✅ All dependencies installed!")

In [ ]:
# Imports
import fitz  # PyMuPDF
import pytesseract
import pandas as pd
from pydantic import BaseModel
from typing import List, Dict, Optional, Any, Tuple
import re
from dataclasses import dataclass, field
from enum import Enum
import cv2
import numpy as np
from collections import Counter

print(f"✅ PyMuPDF: {fitz.version}")
print(f"✅ pandas: {pd.__version__}")
print(f"✅ OpenCV: {cv2.__version__}")
print("✅ All imports successful!")

## 2. Upload Files

In [ ]:
from google.colab import files

PDF_FILE = None
EXCEL_FILE = None

print("📤 Upload testFileQC.pdf AND Appraisal QC Checklist (2).xlsx:")
uploaded = files.upload()

for filename, filedata in uploaded.items():
    with open(filename, "wb") as f:
        f.write(filedata)
    
    if filename.lower().endswith(".pdf"):
        PDF_FILE = filename
        print(f"✅ PDF: {filename}")
    elif filename.lower().endswith(".xlsx"):
        EXCEL_FILE = filename
        print(f"✅ Excel: {filename}")

if not PDF_FILE:
    print("⚠️ No PDF detected. Please re-run.")
if not EXCEL_FILE:
    print("⚠️ No Excel detected. Please re-run.")

## 3. Define Enhanced Data Models

In [ ]:
class RuleCategory(str, Enum):
    SUBJECT = "Subject"
    CONTRACT = "Contract"
    NEIGHBORHOOD = "Neighborhood"
    SITE = "Site"
    IMPROVEMENTS = "Improvements"
    SALES_COMPARISON = "Sales Comparison"
    RECONCILIATION = "Reconciliation"
    PHOTOS = "Photos"
    ADDENDUM = "Addendum"
    SIGNATURE = "Signature"
    OTHER = "Other"


@dataclass
class QCRule:
    item_number: str
    field_name: str
    rule_description: str
    rejection_criteria: str
    category: RuleCategory
    keywords: List[str] = field(default_factory=list)
    patterns: List[str] = field(default_factory=list)
    requires_cv: bool = False  # NEW: Flag for CV-based rules


class ExtractionStatus(str, Enum):
    FOUND = "found"
    MISSING = "missing"
    PARTIAL = "partial"
    INVALID = "invalid"
    CV_SKIPPED = "cv_skipped"  # NEW: For photo rules without model


class ExtractedField(BaseModel):
    rule_id: str
    field_name: str
    category: str
    status: ExtractionStatus
    extracted_value: Optional[str] = None
    page_found: Optional[int] = None
    confidence: float = 0.0
    rule_passed: bool = False
    rejection_reason: Optional[str] = None
    cv_note: Optional[str] = None  # NEW: For CV analysis notes


class SegregatedData(BaseModel):
    subject_fields: List[ExtractedField] = []
    contract_fields: List[ExtractedField] = []
    neighborhood_fields: List[ExtractedField] = []
    site_fields: List[ExtractedField] = []
    improvements_fields: List[ExtractedField] = []
    sales_comparison_fields: List[ExtractedField] = []
    reconciliation_fields: List[ExtractedField] = []
    photo_fields: List[ExtractedField] = []
    addendum_fields: List[ExtractedField] = []
    signature_fields: List[ExtractedField] = []
    other_fields: List[ExtractedField] = []


class ConvincedScore(BaseModel):
    total_rules: int
    rules_passed: int
    rules_failed: int
    rules_partial: int
    rules_missing: int
    cv_rules_skipped: int  # NEW: Count of CV rules skipped
    overall_score: float
    category_scores: Dict[str, float] = {}
    confidence_level: str
    summary: str

print("✅ Data models defined!")

## 4. Parse Excel Rules

In [ ]:
def parse_excel_rules(excel_path: str) -> List[QCRule]:
    """Parse QC checklist with CV flag detection."""
    df = pd.read_excel(excel_path, header=None)
    rules = []
    current_category = RuleCategory.OTHER
    
    # Category keywords
    category_keywords = {
        "subject": RuleCategory.SUBJECT,
        "contract": RuleCategory.CONTRACT,
        "neighborhood": RuleCategory.NEIGHBORHOOD,
        "site": RuleCategory.SITE,
        "improvement": RuleCategory.IMPROVEMENTS,
        "sales comparison": RuleCategory.SALES_COMPARISON,
        "reconciliation": RuleCategory.RECONCILIATION,
        "photo": RuleCategory.PHOTOS,
        "addendum": RuleCategory.ADDENDUM,
        "signature": RuleCategory.SIGNATURE,
    }
    
    # CV-requiring keywords
    cv_keywords = ["photo", "image", "picture", "visual", "condition", "quality rating"]
    
    for idx, row in df.iterrows():
        if idx < 2:
            continue
        
        item_num = str(row.iloc[0]) if pd.notna(row.iloc[0]) else ""
        field_name = str(row.iloc[1]) if pd.notna(row.iloc[1]) else ""
        rule_desc = str(row.iloc[2]) if pd.notna(row.iloc[2]) else ""
        rejection = str(row.iloc[3]) if pd.notna(row.iloc[3]) else ""
        
        # Detect category
        full_text = f"{field_name} {rule_desc}".lower()
        for keyword, cat in category_keywords.items():
            if keyword in full_text and not item_num.replace('.', '').isdigit():
                current_category = cat
                break
        
        # Process rule rows
        if item_num and (item_num.replace('.', '').isdigit() or len(item_num) <= 3):
            keywords = [w.lower() for w in field_name.split() if len(w) > 3]
            
            # Check if CV is required
            requires_cv = any(kw in full_text for kw in cv_keywords) or current_category == RuleCategory.PHOTOS
            
            patterns = []
            if field_name:
                pattern = re.escape(field_name.lower()).replace(r'\ ', r'\s*')
                patterns.append(pattern)
            
            rule = QCRule(
                item_number=item_num,
                field_name=field_name.strip(),
                rule_description=rule_desc.strip(),
                rejection_criteria=rejection.strip(),
                category=current_category,
                keywords=keywords,
                patterns=patterns,
                requires_cv=requires_cv
            )
            rules.append(rule)
    
    return rules


if EXCEL_FILE:
    qc_rules = parse_excel_rules(EXCEL_FILE)
    cv_rules = sum(1 for r in qc_rules if r.requires_cv)
    print(f"📋 Loaded {len(qc_rules)} rules ({cv_rules} require CV)")
    
    category_counts = Counter(r.category.value for r in qc_rules)
    print("\n📊 Rules by Category:")
    for cat, count in category_counts.most_common():
        print(f"   {cat}: {count}")
else:
    print("❌ Excel file not uploaded!")

## 5. Extract PDF Text and Images

In [ ]:
def extract_pdf_pages(pdf_path: str) -> Tuple[Dict[int, str], Dict[int, List]]:
    """Extract both text and images from PDF."""
    doc = fitz.open(pdf_path)
    page_index = {}
    page_images = {}
    
    for i in range(len(doc)):
        page = doc[i]
        
        # Extract text
        text = page.get_text()
        page_index[i + 1] = text
        
        # Extract images
        images = []
        image_list = page.get_images(full=True)
        for img_idx, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to numpy array
            nparr = np.frombuffer(image_bytes, np.uint8)
            img_np = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            if img_np is not None:
                images.append(img_np)
        
        page_images[i + 1] = images
    
    doc.close()
    return page_index, page_images


if PDF_FILE:
    page_index, page_images = extract_pdf_pages(PDF_FILE)
    total_chars = sum(len(t) for t in page_index.values())
    total_images = sum(len(imgs) for imgs in page_images.values())
    
    print(f"📄 Extracted {len(page_index)} pages")
    print(f"📝 Total characters: {total_chars:,}")
    print(f"🖼️ Total images: {total_images}")
    
    # Preview
    print("\n--- Page 1 Preview ---")
    print(page_index[1][:800])
    print("...")
else:
    print("❌ PDF not uploaded!")

## 6. Enhanced Field Extraction with Better Patterns

In [ ]:
def extract_multiline_address(text: str) -> Optional[str]:
    """Extract property address across multiple lines."""
    # Pattern: Street address, City, State Zip
    pattern = r'(\d+\s+[A-Za-z0-9\s,\.]+?)\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\s*,\s*([A-Z]{2})\s+(\d{5})'
    match = re.search(pattern, text)
    if match:
        street, city, state, zip_code = match.groups()
        return f"{street.strip()}, {city.strip()}, {state} {zip_code}"
    return None


def extract_with_enhanced_patterns(page_index: Dict[int, str], rule: QCRule) -> Optional[Tuple[str, int]]:
    """Enhanced extraction with rule-specific patterns."""
    field_lower = rule.field_name.lower()
    
    # Special patterns for commonly failing fields
    special_patterns = {
        "property address": [
            (r"Property\s+Address[:\s]*(.*?)(?:City|\n)", re.IGNORECASE),
            (r"Subject\s+Property[:\s]*(\d+[^\n]+)", re.IGNORECASE),
        ],
        "city": [
            (r"City[:\s]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)", re.IGNORECASE),
        ],
        "state": [
            (r"State[:\s]*([A-Z]{2})", 0),
        ],
        "zip": [
            (r"Zip(?:\s*Code)?[:\s]*(\d{5}(?:-\d{4})?)", re.IGNORECASE),
        ],
        "county": [
            (r"County[:\s]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)", re.IGNORECASE),
        ],
        "pud": [
            (r"(?:PUD|HOA)[:\s]*(.*?)(?:per\s+year|per\s+month|\n)", re.IGNORECASE),
        ],
        "offered for sale": [
            (r"currently\s+offered.*?12\s+months[:\s]*(Yes|No)", re.IGNORECASE | re.DOTALL),
        ],
        "did.*analyze.*contract": [
            (r"(?:Did|Did\s+not)\s+analyze.*?contract[:\s]*(Yes|No|X)", re.IGNORECASE),
        ],
        "zoning": [
            (r"Zoning\s+(?:Classification|Description)[:\s]*([^\n]+)", re.IGNORECASE),
        ],
        "utilities": [
            (r"Utilities[^\n]*?(?:Type|Public|Off(?:-|\s)Site)[:\s]*([^\n]+)", re.IGNORECASE),
        ],
        "design.*style": [
            (r"Design\s*\(?\s*Style\s*\)?[:\s]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)", re.IGNORECASE),
        ],
        "year\s+built": [
            (r"Year\s+Built[:\s]*(\d{4})", re.IGNORECASE),
        ],
        "effective\s+age": [
            (r"Effective\s+Age[:\s]*(\d+)", re.IGNORECASE),
        ],
        "above.*grade.*room": [
            (r"Above\s+(?:Grade|Gr\.?).*?Room.*?Count.*?GLA[:\s]*(\d+\s*/\s*\d+)", re.IGNORECASE | re.DOTALL),
        ],
        "basement.*finish": [
            (r"Basement[^\n]*?(?:Finish|Below\s+Grade)[:\s]*([^\n]+)", re.IGNORECASE),
        ],
    }
    
    # Check field-specific patterns first
    for pattern_key, patterns in special_patterns.items():
        if re.search(pattern_key, field_lower):
            for page_num, text in page_index.items():
                for pattern, flags in patterns:
                    match = re.search(pattern, text, flags if isinstance(flags, int) else 0)
                    if match:
                        value = match.group(1).strip()
                        if value and len(value) > 1:
                            return (value, page_num)
    
    # Fallback to general field name search
    for page_num, text in page_index.items():
        # Try multiple patterns
        patterns = [
            rf"{re.escape(rule.field_name)}[:\s]*([^\n]+)",
            rf"{re.escape(rule.field_name)}\s*[:\.]?\s*([^\n]+)",
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                value = match.group(1).strip()
                if value and len(value) > 1 and value.lower() not in ['nan', 'n/a', 'none']:
                    return (value, page_num)
    
    return None

print("✅ Enhanced extraction patterns loaded")

## 7. Soft CV Integration for Photo Rules

In [ ]:
def analyze_image_basic(img: np.ndarray) -> Dict[str, Any]:
    """Basic CV analysis without trained models."""
    if img is None or img.size == 0:
        return {"error": "Invalid image"}
    
    # Basic image properties
    height, width = img.shape[:2]
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Calculate brightness
    brightness = np.mean(gray)
    
    # Calculate sharpness (Laplacian variance)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    # Edge detection for complexity
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.count_nonzero(edges) / (width * height)
    
    # Color analysis
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    saturation = np.mean(hsv[:, :, 1])
    
    return {
        "dimensions": f"{width}x{height}",
        "brightness": round(brightness, 2),
        "sharpness": round(laplacian_var, 2),
        "edge_density": round(edge_density, 4),
        "saturation": round(saturation, 2),
        "quality_note": "Blurry" if laplacian_var < 100 else "Sharp",
        "lighting_note": "Dark" if brightness < 80 else "Well-lit" if brightness > 150 else "Moderate"
    }


def check_photo_rules_soft(page_images: Dict[int, List]) -> str:
    """Soft check for photo presence and quality."""
    total_images = sum(len(imgs) for imgs in page_images.values())
    
    if total_images == 0:
        return "⚠️ No images found in PDF. Manual review required."
    
    # Analyze first few images
    notes = []
    analyzed = 0
    for page_num, images in page_images.items():
        for img in images[:3]:  # Analyze up to 3 per page
            if analyzed >= 5:  # Limit total analysis
                break
            analysis = analyze_image_basic(img)
            if "error" not in analysis:
                notes.append(f"Image {analyzed+1}: {analysis['quality_note']}, {analysis['lighting_note']}")
            analyzed += 1
    
    summary = f"Found {total_images} images. Sample analysis: " + "; ".join(notes[:3])
    return summary

print("✅ CV analysis functions ready")

## 8. Apply Rules with Enhanced Extraction

In [ ]:
def search_for_field_enhanced(
    page_index: Dict[int, str],
    page_images: Dict[int, List],
    rule: QCRule
) -> ExtractedField:
    """Enhanced field search with CV support."""
    
    # CV-based rules get soft treatment
    if rule.requires_cv:
        cv_note = check_photo_rules_soft(page_images)
        return ExtractedField(
            rule_id=rule.item_number,
            field_name=rule.field_name,
            category=rule.category.value,
            status=ExtractionStatus.CV_SKIPPED,
            extracted_value=None,
            page_found=None,
            confidence=0.5,  # Soft - assume partial pass
            rule_passed=True,  # Soft - don't fail CV rules
            rejection_reason=None,
            cv_note=f"Manual review recommended. {cv_note}"
        )
    
    # Non-CV rules - STRICT extraction
    result = extract_with_enhanced_patterns(page_index, rule)
    
    if result:
        value, page_num = result
        
        # Validate the value
        if value.lower() in ['nan', 'none', 'n/a', '']:
            status = ExtractionStatus.INVALID
            confidence = 0.3
            rule_passed = False
        else:
            status = ExtractionStatus.FOUND
            # Higher confidence for better matches
            confidence = 0.9 if len(value) > 3 else 0.7
            rule_passed = True
        
        return ExtractedField(
            rule_id=rule.item_number,
            field_name=rule.field_name,
            category=rule.category.value,
            status=status,
            extracted_value=value[:200],
            page_found=page_num,
            confidence=confidence,
            rule_passed=rule_passed,
            rejection_reason=None if rule_passed else f"Invalid value: {rule.rejection_criteria}",
            cv_note=None
        )
    else:
        # Not found - STRICT fail for non-CV rules
        return ExtractedField(
            rule_id=rule.item_number,
            field_name=rule.field_name,
            category=rule.category.value,
            status=ExtractionStatus.MISSING,
            extracted_value=None,
            page_found=None,
            confidence=0.0,
            rule_passed=False,
            rejection_reason=f"Field missing: {rule.rejection_criteria}" if rule.rejection_criteria else "Field not found in PDF",
            cv_note=None
        )


def apply_rules_enhanced(
    page_index: Dict[int, str],
    page_images: Dict[int, List],
    rules: List[QCRule]
) -> SegregatedData:
    """Apply all rules with enhanced extraction."""
    segregated = SegregatedData()
    
    category_map = {
        RuleCategory.SUBJECT: segregated.subject_fields,
        RuleCategory.CONTRACT: segregated.contract_fields,
        RuleCategory.NEIGHBORHOOD: segregated.neighborhood_fields,
        RuleCategory.SITE: segregated.site_fields,
        RuleCategory.IMPROVEMENTS: segregated.improvements_fields,
        RuleCategory.SALES_COMPARISON: segregated.sales_comparison_fields,
        RuleCategory.RECONCILIATION: segregated.reconciliation_fields,
        RuleCategory.PHOTOS: segregated.photo_fields,
        RuleCategory.ADDENDUM: segregated.addendum_fields,
        RuleCategory.SIGNATURE: segregated.signature_fields,
        RuleCategory.OTHER: segregated.other_fields,
    }
    
    for rule in rules:
        extracted = search_for_field_enhanced(page_index, page_images, rule)
        target_list = category_map.get(rule.category, segregated.other_fields)
        target_list.append(extracted)
    
    return segregated


# Apply rules
if PDF_FILE and EXCEL_FILE:
    print("🔍 Applying enhanced rules...")
    segregated_data = apply_rules_enhanced(page_index, page_images, qc_rules)
    print("✅ Rules applied!")
else:
    print("❌ Upload both files first")

## 9. Calculate Enhanced Convinced Score

In [ ]:
def calculate_convinced_score_enhanced(segregated: SegregatedData) -> ConvincedScore:
    """Calculate score with CV-skipped handling."""
    all_fields = (
        segregated.subject_fields + segregated.contract_fields +
        segregated.neighborhood_fields + segregated.site_fields +
        segregated.improvements_fields + segregated.sales_comparison_fields +
        segregated.reconciliation_fields + segregated.photo_fields +
        segregated.addendum_fields + segregated.signature_fields +
        segregated.other_fields
    )
    
    total = len(all_fields)
    passed = sum(1 for f in all_fields if f.rule_passed)
    failed = sum(1 for f in all_fields if not f.rule_passed and f.status not in [ExtractionStatus.MISSING, ExtractionStatus.CV_SKIPPED])
    missing = sum(1 for f in all_fields if f.status == ExtractionStatus.MISSING)
    cv_skipped = sum(1 for f in all_fields if f.status == ExtractionStatus.CV_SKIPPED)
    partial = sum(1 for f in all_fields if f.status == ExtractionStatus.PARTIAL)
    
    # Score calculation: CV-skipped rules count as 0.5 (soft treatment)
    weighted_sum = passed * 1.0 + cv_skipped * 0.5 + partial * 0.5
    overall_score = (weighted_sum / total) * 100 if total > 0 else 0.0
    
    # Category scores
    category_scores = {}
    for cat_name, fields in {
        "Subject": segregated.subject_fields,
        "Contract": segregated.contract_fields,
        "Neighborhood": segregated.neighborhood_fields,
        "Site": segregated.site_fields,
        "Improvements": segregated.improvements_fields,
        "Sales Comparison": segregated.sales_comparison_fields,
        "Photos": segregated.photo_fields,
    }.items():
        if fields:
            cat_passed = sum(1 for f in fields if f.rule_passed)
            cat_cv = sum(1 for f in fields if f.status == ExtractionStatus.CV_SKIPPED)
            category_scores[cat_name] = round(((cat_passed + cat_cv * 0.5) / len(fields)) * 100, 1)
    
    confidence_level = "High" if overall_score >= 80 else "Medium" if overall_score >= 50 else "Low"
    
    summary = f"Analyzed {total} rules: {passed} PASSED, {failed} FAILED, {missing} MISSING, {cv_skipped} CV_SOFT. Score: {overall_score:.1f}% ({confidence_level})"
    
    return ConvincedScore(
        total_rules=total,
        rules_passed=passed,
        rules_failed=failed,
        rules_partial=partial,
        rules_missing=missing,
        cv_rules_skipped=cv_skipped,
        overall_score=round(overall_score, 1),
        category_scores=category_scores,
        confidence_level=confidence_level,
        summary=summary
    )


if 'segregated_data' in dir():
    convinced_score = calculate_convinced_score_enhanced(segregated_data)
    print("\n" + "="*60)
    print("📊 CONVINCED SCORE")
    print("="*60)
    print(f"\n{convinced_score.summary}")
    print(f"\n📈 Overall: {convinced_score.overall_score:.1f}%")
    print(f"⚠️ Note: {convinced_score.cv_rules_skipped} photo rules given soft pass (manual review recommended)")

## 10. Display Results by Category

In [ ]:
def display_category(title: str, fields: List[ExtractedField]):
    if not fields:
        return
    
    passed = sum(1 for f in fields if f.rule_passed)
    print(f"\n{'='*60}")
    print(f"📁 {title.upper()} ({passed}/{len(fields)} passed)")
    print(f"{'='*60}")
    
    for f in fields:
        icon = "✅" if f.rule_passed else "❌" if f.status == ExtractionStatus.MISSING else "⚠️"
        if f.status == ExtractionStatus.CV_SKIPPED:
            icon = "🔷"  # Blue diamond for CV soft-pass
        
        print(f"\n{icon} [{f.rule_id}] {f.field_name}")
        print(f"   Status: {f.status.value} | Confidence: {f.confidence:.0%}")
        
        if f.extracted_value:
            val = f.extracted_value[:80] + "..." if len(f.extracted_value) > 80 else f.extracted_value
            print(f"   Value: {val}")
        if f.page_found:
            print(f"   Page: {f.page_found}")
        if f.cv_note:
            print(f"   💡 CV Note: {f.cv_note}")
        if f.rejection_reason:
            print(f"   ⚠️ Issue: {f.rejection_reason}")


if 'segregated_data' in dir():
    display_category("Subject", segregated_data.subject_fields)
    display_category("Contract", segregated_data.contract_fields)
    display_category("Neighborhood", segregated_data.neighborhood_fields)
    display_category("Site", segregated_data.site_fields)
    display_category("Improvements", segregated_data.improvements_fields)
    display_category("Sales Comparison", segregated_data.sales_comparison_fields)
    display_category("Photos", segregated_data.photo_fields)

## 11. Category Scores Visualization

In [ ]:
if 'convinced_score' in dir():
    print("\n" + "="*60)
    print("📈 CATEGORY SCORES")
    print("="*60)
    
    for cat, score in sorted(convinced_score.category_scores.items(), key=lambda x: -x[1]):
        bar = "█" * int(score / 5) + "░" * (20 - int(score / 5))
        status = "✅" if score >= 70 else "⚠️" if score >= 40 else "❌"
        print(f"{status} {cat:20} [{bar}] {score:.1f}%")

## 12. Final Summary

In [ ]:
if 'convinced_score' in dir():
    print("\n" + "="*70)
    print(" " * 20 + "📋 FINAL SUMMARY")
    print("="*70)
    
    print(f"\n📄 Document: {PDF_FILE}")
    print(f"📊 Rules: {convinced_score.total_rules} total")
    print(f"📝 Pages: {len(page_index)}")
    print(f"🖼️ Images: {sum(len(imgs) for imgs in page_images.values())}")
    
    print("\n" + "-"*70)
    print("RESULTS:")
    print("-"*70)
    print(f"   ✅ Passed:        {convinced_score.rules_passed}")
    print(f"   ❌ Failed:        {convinced_score.rules_failed}")
    print(f"   ⚠️ Missing:       {convinced_score.rules_missing}")
    print(f"   🔷 CV Soft-Pass:  {convinced_score.cv_rules_skipped}")
    
    score = convinced_score.overall_score
    grade = "A" if score >= 80 else "B" if score >= 70 else "C" if score >= 60 else "D" if score >= 50 else "F"
    
    print("\n" + "-"*70)
    print(f"   🎯 Overall Score: {score:.1f}% (Grade: {grade})")
    print(f"   📊 Confidence: {convinced_score.confidence_level}")
    
    print("\n" + "-"*70)
    print("NOTES:")
    print("-"*70)
    print("   • Photo-based rules given soft treatment (no trained CV model)")
    print("   • Non-CV rules enforced strictly per OPUS guidelines")
    print("   • Manual review recommended for all CV-skipped items")
    
    print("\n" + "="*70)
    print("✅ Analysis Complete!")
    print("="*70)